# Modelo 1 — Etapa 1: Construção da Tabela Mestra (Feature Store)
## Metodologia CRISP-DM: Fases 1, 2 e 3

Este notebook implementa a estruturação da base analítica contínua para predição de falhas em Portas de Plataforma (PSD) da Linha 4-Amarela do Metrô de São Paulo.

---

### Fase 1: Entendimento do Negócio (Business Understanding)
* **Objetivo:** Antecipar ordens de manutenção corretiva emergencial (ZM01) com 7 dias de antecedência para reduzir paradas não programadas e atrasos no carrossel de trens.
* **Abordagem:** Modelagem preditiva supervisionada no nível porta-dia com horizonte móvel $[D+1, D+7]$.

---

### Fase 2: Entendimento dos Dados (Data Understanding)
* **Fontes Ingeridas:** Logs consolidados de 742k alarmes de telemetria das 11 estações e ordens de serviço SAP (ZM01, ZM02 e ZM03).
* **Mapeamento de Hipóteses:** Isolamento de anomalias críticas ($\ge 15	ext{s}$ e $\ge 30	ext{s}$), falhas mecânicas (`RCE`, `FO`, `LSD`) e modelagem do ciclo de fadiga pós-manutenção (30 a 90 dias).

---

### Fase 3: Preparação dos Dados (Data Preparation)
* Construção da grade diária cartesiana (526 portas $	imes$ 1.155 dias).
* Cálculo de lags temporais com deslocamento estritamente causal (`shift(1)`) para blindagem contra vazamento de dados (*data leakage*).
* Definição do target causal `target_falha_7d`.



In [1]:
import pandas as pd
import numpy as np
import re
import os
import warnings
from datetime import timedelta

warnings.filterwarnings('ignore')

print("Bibliotecas carregadas com sucesso!")


Bibliotecas carregadas com sucesso!


## 1. Data Understanding: Ingestão e Sanitização da Base de Alarmes


In [2]:
path_alarmes = "../../data/Alarmes/alarmes_trata_limpos_consolidado.csv"
if not os.path.exists(path_alarmes):
    path_alarmes = "data/Alarmes/alarmes_trata_limpos_consolidado.csv"

df_alarmes = pd.read_csv(path_alarmes, low_memory=False)

if 'estacao' not in df_alarmes.columns and 'Estação' in df_alarmes.columns:
    df_alarmes['estacao'] = df_alarmes['Estação']
if 'porta' not in df_alarmes.columns and 'Porta' in df_alarmes.columns:
    df_alarmes['porta'] = df_alarmes['Porta']

df_alarmes['estacao'] = df_alarmes['estacao'].fillna(df_alarmes.get('Estação', np.nan))
df_alarmes['porta'] = df_alarmes['porta'].fillna(df_alarmes.get('Porta', np.nan))

def extrair_plt(tag):
    m = re.search(r'_([A-Z]{3})(\d)_', str(tag))
    return f"P{m.group(2)}" if m else "P1"

if 'plataforma' not in df_alarmes.columns:
    df_alarmes['plataforma'] = df_alarmes['TAG'].apply(extrair_plt)
else:
    df_alarmes['plataforma'] = df_alarmes['plataforma'].fillna(df_alarmes['TAG'].apply(extrair_plt))

col_horario = 'Horário do Alarme' if 'Horário do Alarme' in df_alarmes.columns else 'horario_alarme'
df_alarmes['data_evento'] = pd.to_datetime(df_alarmes[col_horario], errors='coerce').dt.normalize()

df_alarmes = df_alarmes.dropna(subset=['estacao', 'plataforma', 'porta', 'data_evento']).copy()
df_alarmes['porta'] = df_alarmes['porta'].astype(int)
df_alarmes['id_ativo'] = df_alarmes['estacao'] + "_" + df_alarmes['plataforma'] + "_PDM" + df_alarmes['porta'].apply(lambda x: f"{x:02d}")
df_alarmes['tempo_normalizacao_seg'] = pd.to_numeric(df_alarmes['tempo_normalizacao_seg'], errors='coerce').fillna(0)

def extrair_tipo(tag):
    tag = str(tag).strip()
    for t in ['RCE', 'FO', 'FC', 'OA', 'CA', 'LSD', 'STS', 'MDU']:
        if t in tag: return t
    return 'OUTRO'

if 'tipo_alarme' not in df_alarmes.columns:
    df_alarmes['tipo_alarme'] = df_alarmes['TAG'].apply(extrair_tipo)
else:
    df_alarmes['tipo_alarme'] = df_alarmes['tipo_alarme'].fillna(df_alarmes['TAG'].apply(extrair_tipo))

df_alarmes['is_anomalia_15s'] = (df_alarmes['tempo_normalizacao_seg'] >= 15).astype(int)
df_alarmes['is_anomalia_30s'] = (df_alarmes['tempo_normalizacao_seg'] >= 30).astype(int)
df_alarmes['is_rce'] = (df_alarmes['tipo_alarme'] == 'RCE').astype(int)
df_alarmes['is_fo'] = (df_alarmes['tipo_alarme'] == 'FO').astype(int)
df_alarmes['is_lsd'] = (df_alarmes['tipo_alarme'] == 'LSD').astype(int)

print(f"Total de alarmes consolidados: {len(df_alarmes):,} registros.")
print(f"Estações identificadas ({df_alarmes['estacao'].nunique()}): {sorted(df_alarmes['estacao'].unique())}")
print(f"Portas únicas monitoradas: {df_alarmes['id_ativo'].nunique():,} portas.")



Total de alarmes consolidados: 742,336 registros.
Estações identificadas (11): ['BUT', 'FAL', 'FRA', 'FRE', 'HIG', 'LUZ', 'MBI', 'PIN', 'PTA', 'REP', 'VSO']
Portas únicas monitoradas: 526 portas.


## 2. Data Understanding: Ingestão do Histórico de Ordens SAP (ZM01, ZM02, ZM03)


In [3]:
padrao_sap = re.compile(r'SP04-6-PSD-([A-Z]{3})-(?:PLT|P)(\d+)(?:-PP(\d+))?', re.IGNORECASE)

def parse_loc_sap(loc_str):
    match = padrao_sap.search(str(loc_str))
    if match and match.group(3):
        return f"{match.group(1).upper()}_P{match.group(2)}_PDM{int(match.group(3)):02d}"
    return None

p_zm01 = "../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx"
if not os.path.exists(p_zm01): p_zm01 = "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx"
df_zm01 = pd.read_excel(p_zm01)
df_zm01['id_ativo'] = df_zm01['local_de_instalacao'].apply(parse_loc_sap)
df_zm01['data_evento'] = pd.to_datetime(df_zm01['inicio_avaria_datetime'], errors='coerce').dt.normalize()
df_zm01 = df_zm01.dropna(subset=['id_ativo', 'data_evento']).copy()

p_zm02 = "../../data/Corretiva Programada/processados/ZM02_limpa.xlsx"
if not os.path.exists(p_zm02): p_zm02 = "data/Corretiva Programada/processados/ZM02_limpa.xlsx"
df_zm02 = pd.read_excel(p_zm02)
df_zm02['id_ativo'] = df_zm02['local_de_instalacao'].apply(parse_loc_sap)
df_zm02['data_evento'] = pd.to_datetime(df_zm02['inicio_avaria_datetime'], errors='coerce').dt.normalize()
df_zm02 = df_zm02.dropna(subset=['id_ativo', 'data_evento']).copy()

p_zm03 = "../../data/Preventiva/preventiva_zm03_limpa.csv"
if not os.path.exists(p_zm03): p_zm03 = "data/Preventiva/preventiva_zm03_limpa.csv"
df_zm03 = pd.read_csv(p_zm03)
df_zm03['data_evento'] = pd.to_datetime(df_zm03['DT_Inicio'], errors='coerce').dt.normalize()
df_zm03 = df_zm03.dropna(subset=['Estacao', 'data_evento']).copy()
df_zm03['estacao'] = df_zm03['Estacao']

print(f"Ordens ZM01 com porta identificada: {len(df_zm01):,}")
print(f"Ordens ZM02 com porta identificada: {len(df_zm02):,}")
print(f"Ordens ZM03 com estação identificada: {len(df_zm03):,}")



Ordens ZM01 com porta identificada: 9,357
Ordens ZM02 com porta identificada: 1,010
Ordens ZM03 com estação identificada: 569


## 3. Data Preparation: Construção da Matriz Contínua Diária (Grade Cartesiana)


In [4]:
ativos_unicos = df_alarmes[['id_ativo', 'estacao', 'plataforma', 'porta']].drop_duplicates().sort_values('id_ativo').reset_index(drop=True)
data_min = pd.to_datetime('2023-01-01')
data_max = pd.to_datetime('2026-02-28')
datas_todas = pd.date_range(start=data_min, end=data_max, freq='D')

grade_base = []
for _, row in ativos_unicos.iterrows():
    df_temp = pd.DataFrame({'data_ref': datas_todas})
    df_temp['id_ativo'] = row['id_ativo']
    df_temp['estacao'] = row['estacao']
    df_temp['plataforma'] = row['plataforma']
    df_temp['porta'] = row['porta']
    grade_base.append(df_temp)

df_mestra = pd.concat(grade_base, ignore_index=True)
df_mestra = df_mestra.sort_values(['id_ativo', 'data_ref']).reset_index(drop=True)

print(f"Grade contínua diária gerada: {len(df_mestra):,} linhas ({len(ativos_unicos)} portas x {len(datas_todas)} dias).")



Grade contínua diária gerada: 607,530 linhas (526 portas x 1155 dias).


## 4. Data Preparation: Engenharia de Features de Telemetria com Lags Causais Estritos


In [5]:
df_alarmes_agg = df_alarmes.groupby(['id_ativo', 'data_evento']).agg(
    qtd_alarmes_dia=('tempo_normalizacao_seg', 'count'),
    qtd_anomalias_15s_dia=('is_anomalia_15s', 'sum'),
    qtd_anomalias_30s_dia=('is_anomalia_30s', 'sum'),
    qtd_rce_dia=('is_rce', 'sum'),
    qtd_fo_dia=('is_fo', 'sum'),
    qtd_lsd_dia=('is_lsd', 'sum'),
    tempo_parada_total_seg_dia=('tempo_normalizacao_seg', 'sum'),
    max_tempo_normalizacao_dia=('tempo_normalizacao_seg', 'max')
).reset_index()

df_alarmes_agg.rename(columns={'data_evento': 'data_ref'}, inplace=True)
df_mestra = pd.merge(df_mestra, df_alarmes_agg, on=['id_ativo', 'data_ref'], how='left')

cols_alarmes = ['qtd_alarmes_dia', 'qtd_anomalias_15s_dia', 'qtd_anomalias_30s_dia', 'qtd_rce_dia', 'qtd_fo_dia', 'qtd_lsd_dia', 'tempo_parada_total_seg_dia', 'max_tempo_normalizacao_dia']
df_mestra[cols_alarmes] = df_mestra[cols_alarmes].fillna(0)

# Agrupamento causal por id_ativo
grouped = df_mestra.groupby('id_ativo')

# Lags causais estritos (.shift(1))
df_mestra['qtd_alarmes_1d'] = grouped['qtd_alarmes_dia'].shift(1).fillna(0)
df_mestra['qtd_alarmes_7d'] = grouped['qtd_alarmes_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)
df_mestra['qtd_alarmes_14d'] = grouped['qtd_alarmes_dia'].shift(1).rolling(14, min_periods=1).sum().fillna(0)

df_mestra['aceleracao_alarmes_7d'] = df_mestra['qtd_alarmes_7d'] - (df_mestra['qtd_alarmes_14d'] - df_mestra['qtd_alarmes_7d'])
df_mestra['qtd_anomalias_15s_7d'] = grouped['qtd_anomalias_15s_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)
df_mestra['qtd_anomalias_30s_14d'] = grouped['qtd_anomalias_30s_dia'].shift(1).rolling(14, min_periods=1).sum().fillna(0)

df_mestra['qtd_rce_7d'] = grouped['qtd_rce_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)
df_mestra['qtd_fo_7d'] = grouped['qtd_fo_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)
df_mestra['qtd_lsd_7d'] = grouped['qtd_lsd_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)

df_mestra['tempo_parada_acumulado_7d'] = grouped['tempo_parada_total_seg_dia'].shift(1).rolling(7, min_periods=1).sum().fillna(0)
df_mestra['max_tempo_normalizacao_7d'] = grouped['max_tempo_normalizacao_dia'].shift(1).rolling(7, min_periods=1).max().fillna(0)
df_mestra['tempo_medio_por_alarme_7d'] = df_mestra['tempo_parada_acumulado_7d'] / (df_mestra['qtd_alarmes_7d'] + 1)
df_mestra['proporcao_alarmes_graves_7d'] = df_mestra['qtd_anomalias_30s_14d'] / (df_mestra['qtd_alarmes_14d'] + 1)

print("Features de telemetria e severidade calculadas com sucesso.")



Features de telemetria e severidade calculadas com sucesso.


## 5. Data Preparation: Features de Histórico de Manutenção e Ciclo de Fadiga


In [6]:
zm01_events = df_zm01[['id_ativo', 'data_evento']].drop_duplicates().sort_values(['id_ativo', 'data_evento'])
zm01_events.rename(columns={'data_evento': 'data_zm01'}, inplace=True)

df_mestra = pd.merge_asof(
    df_mestra.sort_values('data_ref'),
    zm01_events.sort_values('data_zm01'),
    left_on='data_ref',
    right_on='data_zm01',
    by='id_ativo',
    direction='backward',
    allow_exact_matches=False
)
df_mestra['dias_desde_ultima_zm01'] = (df_mestra['data_ref'] - df_mestra['data_zm01']).dt.days.fillna(999).astype(int)
df_mestra.drop(columns=['data_zm01'], inplace=True)

zm02_events = df_zm02[['id_ativo', 'data_evento']].drop_duplicates().sort_values(['id_ativo', 'data_evento'])
zm02_events.rename(columns={'data_evento': 'data_zm02'}, inplace=True)

df_mestra = pd.merge_asof(
    df_mestra.sort_values('data_ref'),
    zm02_events.sort_values('data_zm02'),
    left_on='data_ref',
    right_on='data_zm02',
    by='id_ativo',
    direction='backward',
    allow_exact_matches=False
)
df_mestra['dias_desde_ultima_zm02'] = (df_mestra['data_ref'] - df_mestra['data_zm02']).dt.days.fillna(999).astype(int)
df_mestra.drop(columns=['data_zm02'], inplace=True)

zm03_events = df_zm03[['estacao', 'data_evento']].drop_duplicates().sort_values(['estacao', 'data_evento'])
zm03_events.rename(columns={'data_evento': 'data_zm03'}, inplace=True)

df_mestra = pd.merge_asof(
    df_mestra.sort_values('data_ref'),
    zm03_events.sort_values('data_zm03'),
    left_on='data_ref',
    right_on='data_zm03',
    by='estacao',
    direction='backward',
    allow_exact_matches=False
)
df_mestra['dias_desde_ultima_zm03'] = (df_mestra['data_ref'] - df_mestra['data_zm03']).dt.days.fillna(999).astype(int)
df_mestra.drop(columns=['data_zm03'], inplace=True)

df_mestra['faixa_risco_zm01'] = np.where((df_mestra['dias_desde_ultima_zm01'] >= 30) & (df_mestra['dias_desde_ultima_zm01'] <= 90), 1, 0)
df_mestra['fator_risco_ciclo_zm01'] = np.exp(-((df_mestra['dias_desde_ultima_zm01'] - 60) ** 2) / (2 * (30 ** 2)))

mapa_idade = {'FRA': 16, 'PTA': 16, 'FAL': 16, 'BUT': 15, 'PIN': 15, 'REP': 15, 'LUZ': 15, 'HIG': 8, 'FRE': 8, 'MBI': 8, 'VSO': 5}
df_mestra['idade_ativo_anos'] = df_mestra['estacao'].map(mapa_idade).fillna(15).astype(int)

print("Features de ciclo SAP e idade calculadas com sucesso.")



Features de ciclo SAP e idade calculadas com sucesso.


## 6. Data Preparation: Construção do Target Causal [D+1, D+7] e Exportação


In [7]:
df_mestra = pd.merge(
    df_mestra,
    zm01_events.rename(columns={'data_zm01': 'data_ref'}).assign(ocorreu_zm01_hoje=1),
    on=['id_ativo', 'data_ref'],
    how='left'
)
df_mestra['ocorreu_zm01_hoje'] = df_mestra['ocorreu_zm01_hoje'].fillna(0).astype(int)

df_mestra_inv = df_mestra.iloc[::-1].copy()
grouped_inv = df_mestra_inv.groupby('id_ativo')
df_mestra_inv['target_falha_7d'] = grouped_inv['ocorreu_zm01_hoje'].shift(1).rolling(7, min_periods=1).max().fillna(0).astype(int)
df_mestra = df_mestra_inv.iloc[::-1].copy()

cols_export = [
    'id_ativo', 'data_ref', 'estacao', 'plataforma', 'porta', 'idade_ativo_anos',
    'dias_desde_ultima_zm01', 'dias_desde_ultima_zm02', 'dias_desde_ultima_zm03',
    'faixa_risco_zm01', 'fator_risco_ciclo_zm01',
    'qtd_alarmes_1d', 'qtd_alarmes_7d', 'qtd_alarmes_14d', 'aceleracao_alarmes_7d',
    'qtd_anomalias_15s_7d', 'qtd_anomalias_30s_14d',
    'qtd_rce_7d', 'qtd_fo_7d', 'qtd_lsd_7d',
    'tempo_parada_acumulado_7d', 'max_tempo_normalizacao_7d',
    'tempo_medio_por_alarme_7d', 'proporcao_alarmes_graves_7d',
    'target_falha_7d'
]

df_mestra = df_mestra[cols_export]

p_out = "../../data/tabela_mestra_bruta.csv"
if not os.path.exists("../../data"): p_out = "data/tabela_mestra_bruta.csv"
df_mestra.to_csv(p_out, index=False)

print(f"Tabela Mestra gerada e salva com sucesso em: {p_out}")
print(f"Dimensões: {df_mestra.shape[0]:,} linhas x {df_mestra.shape[1]} colunas.")
print(f"Prevalência Global do Target: {df_mestra['target_falha_7d'].mean()*100:.2f}%")



Tabela Mestra gerada e salva com sucesso em: ../../data/tabela_mestra_bruta.csv
Dimensões: 607,530 linhas x 25 colunas.
Prevalência Global do Target: 8.49%
